In [1]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *

In [2]:
def verificar_estacionaridade(df, nivel_significancia=0.05):
    """
    Roda o teste Augmented Dickey-Fuller (ADF) em todas as colunas numéricas.
    Retorna um DataFrame com os resultados.
    """
    resultados = []
    
    print(f"Iniciando Teste ADF em {len(df.columns)} variáveis...")
    print("-" * 80)
    
    # Selecionar apenas colunas numéricas (ignorar Data_UTC se estiver como coluna)
    cols_numericas = df.select_dtypes(include=[np.number]).columns
    
    for col in cols_numericas:
        # Limpeza rápida para o teste (ADF não aceita NaNs ou Infinitos)
        serie_limpa = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(serie_limpa) < 20: # Precisa de dados mínimos
            resultados.append({
                'Variavel': col,
                'ADF Statistic': np.nan,
                'P-Valor': np.nan,
                'Resultado': 'Dados Insuficientes'
            })
            continue

        try:
            # Rodando o Teste ADF
            result = adfuller(serie_limpa, autolag='AIC')
            
            p_value = result[1]
            adf_stat = result[0]
            
            # Critério de Decisão
            if p_value < nivel_significancia:
                status = "✅ ESTACIONÁRIA"
            else:
                status = "❌ NÃO ESTACIONÁRIA"
                
            resultados.append({
                'Variavel': col,
                'ADF Statistic': round(adf_stat, 4),
                'P-Valor': round(p_value, 6),
                'Resultado': status
            })
            
        except Exception as e:
            resultados.append({
                'Variavel': col,
                'ADF Statistic': np.nan,
                'P-Valor': np.nan,
                'Resultado': f"Erro: {str(e)}"
            })

    # Criando DataFrame do Relatório
    df_resultado = pd.DataFrame(resultados)
    df_resultado = df_resultado.sort_values(by='P-Valor', ascending=True)
    
    return df_resultado

## 📊 Interpretação dos Resultados do Teste ADF

### ✅ Sinal Verde (ESTACIONÁRIA)

**O que significa:**  
O P-Valor foi menor que 0.05 (ex: 0.000000).

**Ação:**  
Essa variável está perfeita. Ela entra direto no modelo VAR ou Teste de Granger.

**Exemplo Esperado:**  
`btc_log_ret`, `us10y_diff`, `eth_vol_log_ret`. Como aplicamos Log e Diff, 99% das suas features devem cair aqui.

---

### ❌ Sinal Vermelho (NÃO ESTACIONÁRIA)

**O que significa:**  
O P-Valor foi alto (ex: 0.85 ou 0.15). A variável ainda tem tendência (sobe ou desce com o tempo).

**Causa provável:**  
Talvez alguma feature tenha escapado do tratamento (ex: se você acidentalmente colocou o Preço Bruto `btc_price` em vez do retorno).

**Ação:**

- **Se for uma variável bruta (ex: Preço):** Não use no teste de Granger. Use a versão tratada (`_log_ret` ou `_diff`).
  
- **Se for uma variável já tratada (ex: `_diff`) que falhou:** Tente aplicar uma segunda diferença (`.diff().diff()`).

In [3]:
df_adf_report = verificar_estacionaridade(df_features_transformada, nivel_significancia=0.05)
# df_adf_report.to_excel(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/src/output/estacionaridade_report.xlsx", index=False)
df_adf_report

Iniciando Teste ADF em 25 variáveis...
--------------------------------------------------------------------------------


,Variavel,ADF Statistic,P-Valor,Resultado
0,rvi_diff,-15.3796,0.000000,✅ ESTACIONÁRIA
21,vol_acceleration,-15.0780,0.000000,✅ ESTACIONÁRIA
18,flippening_close_diff,-8.6660,0.000000,✅ ESTACIONÁRIA
17,ssr_diff,-9.8437,0.000000,✅ ESTACIONÁRIA
16,total3_log_ret,-9.4219,0.000000,✅ ESTACIONÁRIA
15,vix_log_ret,-18.8494,0.000000,✅ ESTACIONÁRIA
14,us10y_diff,-41.3738,0.000000,✅ ESTACIONÁRIA
13,gold_log_ret,-22.5673,0.000000,✅ ESTACIONÁRIA
12,nasdaq_log_ret,-10.6863,0.000000,✅ ESTACIONÁRIA
22,usdt_log_ret,-10.6218,0.000000,✅ ESTACIONÁRIA
